# 03 — Análise regional dos preços reais da habitação

Este notebook usa a hierarquia oficial NUTS 2024 e os valores em euros constantes de 2025 para responder a cinco perguntas:

1. Que regiões apresentam os preços municipais mais elevados em 2025?
2. Em que regiões cresceram mais os preços acima da inflação entre 2021 e 2025?
3. Que municípios se destacam dentro da sua própria NUTS III?
4. Como varia regionalmente o indicador exploratório preço–rendimento?
5. A relação entre migração e crescimento real é semelhante em todas as regiões?

A unidade de observação continua a ser o município. As estatísticas regionais abaixo são calculadas a partir dos municípios e não correspondem necessariamente a indicadores regionais publicados diretamente pelo INE.


## 1. Preparação

Carregamos a tabela de features produzida pela pipeline. Os códigos geográficos são lidos como texto porque são identificadores, não quantidades.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Não foi possível encontrar a raiz do projeto.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
FEATURE_PATH = PROJECT_ROOT / "data/processed/municipality_year_features_2021_2025.csv"
if not FEATURE_PATH.exists():
    raise FileNotFoundError(
        "Tabela de features não encontrada. Executa primeiro: "
        "python -m src.data.build_cpi && python -m src.features.build_features"
    )

features = pd.read_csv(
    FEATURE_PATH,
    dtype={
        "municipality_code": "string",
        "nuts2_code": "string",
        "nuts3_code": "string",
    },
)
features.head()


## 2. Cobertura regional

Antes da análise confirmamos a granularidade da tabela e contamos os municípios de cada NUTS II. As regiões têm números de municípios muito diferentes; por exemplo, nove na Grande Lisboa e 86 no Norte.


In [ ]:
assert len(features) == 1_540
assert not features.duplicated(["municipality_code", "year"]).any()
assert sorted(features["year"].unique()) == [2021, 2022, 2023, 2024, 2025]
assert features["municipality_code"].nunique() == 308
assert features["nuts2_code"].nunique() == 9
assert features["nuts3_code"].nunique() == 26

municipalities_by_nuts2 = (
    features.drop_duplicates("municipality_code")
    .groupby(["nuts2_code", "nuts2_name"])
    .size()
    .rename("municipalities")
    .sort_values(ascending=False)
    .to_frame()
)
municipalities_by_nuts2


### Como interpretar as medianas regionais

Usaremos principalmente a **mediana dos municípios**. Cada município tem o mesmo peso, independentemente da população ou do número de vendas.

Por exemplo, Lisboa e Arruda dos Vinhos contam como uma observação cada. Esta escolha responde à pergunta “como é o município típico desta região?”, mas não responde à pergunta “qual é o preço típico pago por uma pessoa ou numa transação da região?”. Para essa segunda pergunta precisaríamos de ponderadores adequados, como o número de transações.


## 3. Preços reais por NUTS II em 2025

Como 2025 é o ano base do IPC, os valores nominais e os valores em euros constantes de 2025 coincidem nesse ano. Calculamos a mediana e o intervalo interquartil dos municípios de cada região.


In [ ]:
prices_2025 = features.loc[features["year"].eq(2025)].copy()

regional_price_2025 = (
    prices_2025.groupby(["nuts2_code", "nuts2_name"])
    .agg(
        municipalities=("municipality_code", "size"),
        prices_present=("housing_price_m2_total_2025_eur", "count"),
        median_price_2025_eur_m2=("housing_price_m2_total_2025_eur", "median"),
        q1=("housing_price_m2_total_2025_eur", lambda values: values.quantile(0.25)),
        q3=("housing_price_m2_total_2025_eur", lambda values: values.quantile(0.75)),
    )
    .sort_values("median_price_2025_eur_m2", ascending=False)
)
regional_price_2025


In [ ]:
ax = regional_price_2025.sort_values("median_price_2025_eur_m2")[
    "median_price_2025_eur_m2"
].plot(kind="barh", figsize=(9, 5), color="#2878B5")
ax.set_title("Mediana municipal do preço real por NUTS II, 2025")
ax.set_xlabel("Euros de 2025 por m²")
ax.set_ylabel("NUTS II")
plt.tight_layout()


In [ ]:
regional_price_trend = (
    features.groupby(["year", "nuts2_name"])["housing_price_m2_total_2025_eur"]
    .median()
    .unstack("nuts2_name")
)

ax = regional_price_trend.plot(figsize=(11, 6), marker="o")
ax.set_title("Evolução da mediana municipal do preço em euros de 2025")
ax.set_xlabel("Ano")
ax.set_ylabel("Euros de 2025 por m²")
ax.set_xticks(regional_price_trend.index)
ax.legend(title="NUTS II", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()


### Conclusão — nível dos preços

Em 2025, a Grande Lisboa apresenta a maior mediana municipal, aproximadamente **3 057 €/m²**, seguida do Algarve, com **2 969 €/m²**, e da Península de Setúbal, com **2 508 €/m²**. Centro e Alentejo apresentam as menores medianas, aproximadamente 751 e 818 €/m².

Isto descreve o nível do preço, não a velocidade de crescimento. A Região Autónoma dos Açores tem apenas 16 dos 19 preços municipais disponíveis em 2025; a tabela mostra explicitamente essa diferença entre municípios existentes e preços presentes.


## 4. Crescimento real entre 2021 e 2025

Para cada município com preço disponível nos dois extremos, calculamos:

```text
crescimento real acumulado = (preço real de 2025 / preço real de 2021 - 1) × 100
```

Depois resumimos esses crescimentos municipais através da mediana de cada NUTS II. Não calculamos a variação de uma média regional agregada.


In [ ]:
price_endpoints = features.pivot(
    index=[
        "municipality_code",
        "municipality_name",
        "nuts2_code",
        "nuts2_name",
        "nuts3_code",
        "nuts3_name",
    ],
    columns="year",
    values="housing_price_m2_total_2025_eur",
).dropna(subset=[2021, 2025])

price_endpoints["real_growth_2021_2025_pct"] = (
    price_endpoints[2025].div(price_endpoints[2021]).sub(1).mul(100)
)

regional_real_growth = (
    price_endpoints.groupby(["nuts2_code", "nuts2_name"])[
        "real_growth_2021_2025_pct"
    ]
    .agg(comparable_municipalities="count", median="median", mean="mean")
    .sort_values("median", ascending=False)
)
regional_real_growth


In [ ]:
ax = regional_real_growth.sort_values("median")["median"].plot(
    kind="barh", figsize=(9, 5), color="#F28E2B"
)
ax.set_title("Crescimento real municipal mediano, 2021–2025")
ax.set_xlabel("Crescimento acumulado (%)")
ax.set_ylabel("NUTS II")
plt.tight_layout()


### Conclusão — crescimento real

O maior crescimento real mediano aparece na Região Autónoma da Madeira (**50,3%**), seguida dos Açores (**49,0%**), Península de Setúbal (**47,6%**) e Oeste e Vale do Tejo (**45,9%**). Norte e Centro apresentam as menores medianas, aproximadamente 24,6% e 25,3%.

Uma região pode ter preços baixos e, ainda assim, crescer rapidamente. Por isso, “região mais cara” e “região com maior crescimento” são conclusões diferentes.


## 5. Posição do município dentro da NUTS III

Uma comparação nacional pode esconder diferenças locais. Criamos uma medida relativa:

```text
(preço do município / mediana da respetiva NUTS III - 1) × 100
```

Um resultado de `50%` significa que o município está 50% acima da mediana dos municípios da sua sub-região naquele ano. Um resultado de `-30%` significa que está 30% abaixo.


In [ ]:
prices_2025["nuts3_median_price"] = prices_2025.groupby("nuts3_code")[
    "housing_price_m2_total_2025_eur"
].transform("median")
prices_2025["price_vs_nuts3_median_pct"] = (
    prices_2025["housing_price_m2_total_2025_eur"]
    .div(prices_2025["nuts3_median_price"])
    .sub(1)
    .mul(100)
)

relative_columns = [
    "municipality_name",
    "nuts3_name",
    "nuts2_name",
    "housing_price_m2_total_2025_eur",
    "nuts3_median_price",
    "price_vs_nuts3_median_pct",
]

print("Municípios mais acima da mediana da sua NUTS III:")
display(prices_2025.nlargest(10, "price_vs_nuts3_median_pct")[relative_columns])
print("Municípios mais abaixo da mediana da sua NUTS III:")
display(prices_2025.nsmallest(10, "price_vs_nuts3_median_pct")[relative_columns])


### Conclusão — contexto sub-regional

Vila Real apresenta um preço aproximadamente **183% acima** da mediana municipal do Douro. Bragança, Évora, Coimbra, Covilhã e Viseu também se destacam dentro das respetivas NUTS III. No extremo oposto, Alcoutim está cerca de **68% abaixo** da mediana do Algarve.

Esta medida identifica contraste dentro da região. Não diz que Vila Real é o município mais caro de Portugal, nem prova a causa da diferença. Sub-regiões com poucos municípios também produzem medianas menos estáveis.


## 6. Preço e rendimento por região em 2024

O rendimento só está disponível até 2024. Resumimos o indicador exploratório que compara o preço de um m² com o rendimento anual declarado deduzido de IRS por agregado fiscal.

Uma mediana de `14,6%` significa que, no município mediano da região, o preço de um m² representa 14,6% desse indicador anual de rendimento. Não é uma taxa de esforço habitacional oficial.


In [ ]:
regional_price_income_2024 = (
    features.loc[features["year"].eq(2024)]
    .groupby(["nuts2_code", "nuts2_name"])
    .agg(
        municipalities=("municipality_code", "size"),
        valid=("one_m2_price_as_declared_income_pct", "count"),
        median_price_income_pct=("one_m2_price_as_declared_income_pct", "median"),
        median_real_income=(
            "declared_income_less_irs_per_tax_household_2025_eur",
            "median",
        ),
        median_real_price=("housing_price_m2_total_2025_eur", "median"),
    )
    .sort_values("median_price_income_pct", ascending=False)
)
regional_price_income_2024


In [ ]:
ax = regional_price_income_2024.sort_values("median_price_income_pct")[
    "median_price_income_pct"
].plot(kind="barh", figsize=(9, 5), color="#59A14F")
ax.set_title("Mediana regional do indicador preço–rendimento, 2024")
ax.set_xlabel("Preço de um m² / rendimento anual (%)")
ax.set_ylabel("NUTS II")
plt.tight_layout()


In [ ]:
price_income_correlations = []
for (nuts2_code, nuts2_name), group in features.loc[
    features["year"].eq(2024)
].groupby(["nuts2_code", "nuts2_name"]):
    valid = group[[
        "housing_price_m2_total_2025_eur",
        "declared_income_less_irs_per_tax_household_2025_eur",
    ]].dropna()
    price_income_correlations.append(
        {
            "nuts2_code": nuts2_code,
            "nuts2_name": nuts2_name,
            "municipalities": len(valid),
            "pearson": valid.corr(method="pearson").iloc[0, 1],
            "spearman": valid.corr(method="spearman").iloc[0, 1],
        }
    )

price_income_correlations = pd.DataFrame(price_income_correlations).sort_values(
    "spearman", ascending=False
)
price_income_correlations


### Conclusão — preço e rendimento

Em 2024, o Algarve apresenta a maior mediana do indicador preço–rendimento (**14,6%**), acima da Grande Lisboa (**12,6%**) e da Península de Setúbal (**9,7%**). O Algarve não tem o rendimento mediano mais baixo, mas o seu preço é elevado em relação ao rendimento fiscal observado.

Dentro do Centro e do Norte, preço e rendimento municipal apresentam correlações positivas fortes em 2024. No Algarve, a correlação é próxima de zero. Isto mostra que uma relação nacional pode esconder padrões regionais diferentes. As regiões com apenas nove municípios exigem especial cautela.


## 7. Migração média e crescimento real acumulado

Para cada município calculamos a média da taxa migratória anual entre 2021 e 2025 e comparamo-la com o crescimento real acumulado do preço entre 2021 e 2025.

Esta comparação resume o período, mas continua a ser observacional: não estabelece direção causal e pode ocultar atrasos temporais.


In [ ]:
average_migration = (
    features.groupby(["municipality_code", "municipality_name"])[
        "migration_rate_per_1000"
    ]
    .mean()
    .rename("average_migration_rate_per_1000")
    .reset_index()
)

migration_growth = price_endpoints.reset_index().merge(
    average_migration,
    on=["municipality_code", "municipality_name"],
    validate="one_to_one",
)

national_migration_correlations = migration_growth[[
    "average_migration_rate_per_1000",
    "real_growth_2021_2025_pct",
]].corr()

regional_migration_correlations = []
for (nuts2_code, nuts2_name), group in migration_growth.groupby(
    ["nuts2_code", "nuts2_name"]
):
    pair = group[[
        "average_migration_rate_per_1000",
        "real_growth_2021_2025_pct",
    ]]
    regional_migration_correlations.append(
        {
            "nuts2_code": nuts2_code,
            "nuts2_name": nuts2_name,
            "municipalities": len(pair),
            "pearson": pair.corr(method="pearson").iloc[0, 1],
            "spearman": pair.corr(method="spearman").iloc[0, 1],
        }
    )

print("Correlação nacional de Pearson:", national_migration_correlations.iloc[0, 1])
print(
    "Correlação nacional de Spearman:",
    migration_growth[[
        "average_migration_rate_per_1000",
        "real_growth_2021_2025_pct",
    ]].corr(method="spearman").iloc[0, 1],
)
pd.DataFrame(regional_migration_correlations).sort_values("spearman", ascending=False)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for nuts2_name, group in migration_growth.groupby("nuts2_name"):
    ax.scatter(
        group["average_migration_rate_per_1000"],
        group["real_growth_2021_2025_pct"],
        label=nuts2_name,
        alpha=0.7,
    )
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Migração média e crescimento real do preço, 2021–2025")
ax.set_xlabel("Taxa migratória média anual por 1 000 habitantes")
ax.set_ylabel("Crescimento real acumulado do preço (%)")
ax.legend(title="NUTS II", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()


### Conclusão — migração e crescimento

No conjunto dos 298 municípios comparáveis, a correlação é positiva mas fraca: aproximadamente **0,191 em Pearson** e **0,219 em Spearman**. As correlações regionais variam de positivas a negativas.

A Península de Setúbal apresenta uma associação positiva elevada nesta amostra, enquanto Algarve e Grande Lisboa apresentam associações negativas. Cada uma destas regiões tem poucos municípios, pelo que estes coeficientes são instáveis e não justificam conclusões causais. O resultado reforça a necessidade de controlar simultaneamente ano, região e outras características.


## 8. Conclusões e próximo passo

A análise regional mostra quatro ideias principais:

1. **Nível e crescimento são diferentes.** Grande Lisboa e Algarve têm os preços mais elevados, enquanto Madeira e Açores apresentam o maior crescimento real mediano entre 2021 e 2025.
2. **O contexto local muda a leitura.** Municípios como Vila Real e Évora destacam-se muito mais quando comparados com a própria NUTS III.
3. **Rendimento não explica tudo da mesma maneira.** A relação municipal entre rendimento e preço é forte em algumas regiões e fraca noutras.
4. **A associação com migração não é estável.** O padrão nacional é fraco e as estimativas regionais variam, sobretudo em grupos pequenos.

Limitações principais:

- cinco anos constituem um período curto;
- as medianas dão o mesmo peso a todos os municípios;
- não temos número de vendas para ponderar os preços;
- o IPC é nacional e não mede custo de vida local;
- faltam variáveis sobre construção, oferta, turismo, crédito e características das habitações;
- correlação não demonstra causalidade.

O próximo passo será preparar um modelo de painel simples. Antes de o interpretar, vamos definir claramente a variável-alvo, escolher apenas variáveis disponíveis no mesmo período e incluir efeitos de ano e de região.
